In [32]:
import pandas as pd
import numpy as np
import os

from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

# 데이터 불러오기

In [33]:
# data load
# 모델링에 사용할 데이터를 로드해옵니다.
train = pd.read_csv('./train.csv')
test = pd.read_csv('./test.csv')

In [34]:
# train data에서 결측치가 보입니다.
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58627 entries, 0 to 58626
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   brand         58627 non-null  object 
 1   model         58627 non-null  object 
 2   year          58627 non-null  int64  
 3   transmission  58627 non-null  object 
 4   mileage       57849 non-null  float64
 5   fuelType      58627 non-null  object 
 6   tax           53515 non-null  float64
 7   mpg           53035 non-null  float64
 8   engineSize    58627 non-null  float64
 9   price         58627 non-null  int64  
dtypes: float64(4), int64(2), object(4)
memory usage: 4.5+ MB


In [35]:
# 결측치를 모두 제거합니다.
train = train.dropna()
print(train.isnull().sum())

brand           0
model           0
year            0
transmission    0
mileage         0
fuelType        0
tax             0
mpg             0
engineSize      0
price           0
dtype: int64


In [36]:
# 독립변수로 설정할 train_x에서는 종속변수를 제거합니다.
train_x = train.drop(['price'], axis = 1)
# train_y 변수를 종속변수로 사용하기 위해 price 데이터를 지정하였습니다.
train_y = train['price']

# train_x와 달리 분석에 활용하지 않는 ID 데이터를 제거합니다.
test_x = test.drop('id', axis = 1)

In [37]:
df_brand = pd.read_csv('./brand_model.csv')

In [38]:
df_brand

,brand,model
0,audi,A1
1,audi,A2
2,audi,A3
3,audi,A4
4,audi,A5
...,...,...
190,vw,Tiguan
191,vw,Tiguan Allspace
192,vw,Touareg
193,vw,Touran


# 인코딩

범주형 변수에서 train의 class와 test의 class가 같은지, 다른지 확인

--> 같으면 그대로 진행

--> 다르면 두 가지 경우가 발생


1. 범주형 변수의 class가 정확히 주어진 경우
2. 범주형 변수의 class가 정확히 주어지지 않은 경우

1의 경우에는 정확히 주어진 class를 가지고 인코더에 fit을 진행 후 train, test에 transform을 진행

2의 경우에는 train에는 없지만 test의 있는 class를 Unknown(모른다)라고 라벨링 후 인코더에 fit을 진행 후 train, test에 transform을 진행

여기서는 train의 class와 test의 class가 다른 변수가 'model' 뿐이고, 또한 'model'의 class가 'brand_model.csv'로 주어져 있기 때문에 1의 경우에만 고려하면 됨.

In [39]:
# 인코딩을 진행할 범주형 변수에서 train의 class와 test의 class가 같은지, 다른지 확인
object_features = ['model', 'brand', 'transmission', 'fuelType']

for feature in object_features:
    # train set에 있는 클래스 확인
    train_classes = set(train_x[feature])

    # test set에만 있는 클래스 확인
    test_classes = set(test_x[feature]) - train_classes
    if test_classes:
        print(f'{feature} : {test_classes}')
    else:
        print(f'{feature} : 없음')
    

model : {' Escort', '200', '180', ' Kadjar', ' Veloster', '230', ' Streetka', '220', ' Ranger', ' A2'}
brand : 없음
transmission : 없음
fuelType : 없음


## label encoding

In [40]:
# 인코딩을 진행할 범주형 변수의 class가 정확히 주어진 경우(model.csv파일 존재)는 다음과 같이 사용
from sklearn.preprocessing import LabelEncoder
object_features = ['brand', 'model']

for feature in object_features:
    le = LabelEncoder()
    le = le.fit(df_brand[feature].unique()) # 각 범주형 변수의 주어진 class에 대해 fit
    
    train_x[feature] = le.transform(train_x[feature]) # train_x에 적용
    test_x[feature] = le.transform(test_x[feature]) # test_x에 적용

In [41]:
# 범주형 변수의 class가 정확히 주어지지 않을 경우는 다음과 같이 사용
object_features = ['transmission', 'fuelType']

for feature in object_features:
    # train set에 있는 클래스 확인
    train_classes = set(train_x[feature])

    # test set에만 있는 클래스 확인
    test_classes = set(test_x[feature]) - train_classes
    
    if len(test_classes) != 0: # train에서의 class와 test에서의 class가 다른 경우
        test_x.loc[test_x[feature].isin(test_classes), feature] = 'Unknown' # test에만 있는 class를 unknown으로 치환
        le = LabelEncoder()
        le = le.fit(train_x[feature])
        train_x[feature] = le.transform(train_x[feature])
        le.classes_ = np.append(le.classes_, 'Unknown') # label encoder의 classes에 Unknown 추가 후 test set을 transform
        test_x[feature] = le.transform(test_x[feature])
        print(f'{feature} 컬럼은 test set에만 있는 클래스 {test_classes}를 포함하고 있습니다.')
        
    else: # train에서의 class와 test에서의 class가 같은 경우
        le = LabelEncoder()
        le = le.fit(train_x[feature])
        train_x[feature] = le.transform(train_x[feature])
        test_x[feature] = le.transform(test_x[feature])

In [42]:
train_x

,brand,model,year,transmission,mileage,fuelType,tax,mpg,engineSize
0,3,83,2015,0,18078.0,0,20.0,67.3,1.6
1,4,8,2018,0,4350.0,0,145.0,68.9,2.1
3,6,170,2011,1,74488.0,4,200.0,41.5,1.6
4,4,32,2019,0,608.0,4,145.0,42.2,2.0
5,5,59,2019,1,3972.0,4,145.0,52.3,1.0
...,...,...,...,...,...,...,...,...,...
58622,8,165,2017,1,33610.0,4,20.0,64.2,1.0
58623,6,182,2013,1,57140.0,4,125.0,51.4,1.3
58624,7,52,2017,1,9832.0,4,150.0,55.4,1.4
58625,8,115,2017,1,28984.0,0,145.0,83.1,1.4


In [43]:
test_x

,brand,model,year,transmission,mileage,fuelType,tax,mpg,engineSize
0,4,32,2019,3,14271,4,145,46.3,1.5
1,2,61,2019,1,13451,4,145,58.9,1.0
2,3,164,2016,1,58264,0,30,61.7,1.7
3,1,2,2015,3,6147,4,145,47.9,2.0
4,0,11,2019,1,7000,0,145,51.4,1.6
...,...,...,...,...,...,...,...,...,...
39080,3,164,2016,1,48868,0,30,61.7,1.7
39081,6,33,2019,0,8300,2,135,74.3,1.8
39082,4,54,2019,0,16916,0,145,65.7,2.0
39083,1,5,2014,0,27782,0,160,50.4,3.0


## Onehot encoding


범주형 변수에 대해 get_dummies 변수를 쓰게되면 train에 있는 class랑, test에 있는 class가 다를 경우에는 변수 개수가 달라짐

따라서 label encoding처럼 OneHotEncoding 객채를 생성 후 fit, transform을 해줘야함

만약 train에 있는 class 와 test에 있는 class가 같다면 get_dummies를 사용해도 무방

In [20]:
# from sklearn.preprocessing import OneHotEncoder
# # train에서의 class와 test에서의 class가 다른 경우 OneHotEncoder 사용
# object_features = ['model']

# for feature in object_features:
#     ohe = OneHotEncoder()
#     ohe = ohe.fit(pd.DataFrame(df_brand[feature].unique()))

#     # train 데이터 onehot encoding
#     temp_train = ohe.transform(pd.DataFrame(train_x[feature]))
#     temp_train = temp_train.toarray()
#     feature_names = ohe.get_feature_names_out(input_features=[feature])
#     temp_train = pd.DataFrame(temp_train, columns=feature_names)
#     train_x.reset_index(drop=True, inplace=True)
#     train_x = pd.concat([train_x, temp_train], axis=1)
#     train_x = train_x.drop(feature, axis=1)


#     # test 데이터 onehot encoding
#     temp_test = ohe.transform(pd.DataFrame(test_x[feature]))
#     temp_test = temp_test.toarray()
#     feature_names = ohe.get_feature_names_out(input_features=[feature])
#     temp_test = pd.DataFrame(temp_test, columns=feature_names)
#     test_x = pd.concat([test_x, temp_test], axis=1)
#     test_x = test_x.drop(feature, axis=1)

c:\Users\user\anaconda3\envs\py_397\lib\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but OneHotEncoder was fitted without feature names
  warnings.warn(
c:\Users\user\anaconda3\envs\py_397\lib\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but OneHotEncoder was fitted without feature names
  warnings.warn(


In [21]:
# # train에서의 class와 test에서의 class가 같은 경우 get_dummies 사용
# object_features = ['brand', 'transmission', 'fuelType']

# train_x = pd.get_dummies(train_x, columns=object_features)
# test_x = pd.get_dummies(test_x, columns=object_features)

In [22]:
# train_x

,year,mileage,tax,mpg,engineSize,model_ 1 Series,model_ 2 Series,model_ 3 Series,model_ 4 Series,model_ 5 Series,...,brand_vw,transmission_Automatic,transmission_Manual,transmission_Other,transmission_Semi-Auto,fuelType_Diesel,fuelType_Electric,fuelType_Hybrid,fuelType_Other,fuelType_Petrol
0,2015,18078.0,20.0,67.3,1.6,0.0,0.0,0.0,0.0,0.0,...,False,True,False,False,False,True,False,False,False,False
1,2018,4350.0,145.0,68.9,2.1,0.0,0.0,0.0,0.0,0.0,...,False,True,False,False,False,True,False,False,False,False
2,2011,74488.0,200.0,41.5,1.6,0.0,0.0,0.0,0.0,0.0,...,False,False,True,False,False,False,False,False,False,True
3,2019,608.0,145.0,42.2,2.0,0.0,0.0,0.0,0.0,0.0,...,False,True,False,False,False,False,False,False,False,True
4,2019,3972.0,145.0,52.3,1.0,0.0,0.0,0.0,0.0,0.0,...,False,False,True,False,False,False,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47776,2017,33610.0,20.0,64.2,1.0,0.0,0.0,0.0,0.0,0.0,...,True,False,True,False,False,False,False,False,False,True
47777,2013,57140.0,125.0,51.4,1.3,0.0,0.0,0.0,0.0,0.0,...,False,False,True,False,False,False,False,False,False,True
47778,2017,9832.0,150.0,55.4,1.4,0.0,0.0,0.0,0.0,0.0,...,False,False,True,False,False,False,False,False,False,True
47779,2017,28984.0,145.0,83.1,1.4,0.0,0.0,0.0,0.0,0.0,...,True,False,True,False,False,True,False,False,False,False


In [23]:
# test_x

,year,mileage,tax,mpg,engineSize,model_ 1 Series,model_ 2 Series,model_ 3 Series,model_ 4 Series,model_ 5 Series,...,brand_vw,transmission_Automatic,transmission_Manual,transmission_Other,transmission_Semi-Auto,fuelType_Diesel,fuelType_Electric,fuelType_Hybrid,fuelType_Other,fuelType_Petrol
0,2019,14271,145,46.3,1.5,0.0,0.0,0.0,0.0,0.0,...,False,False,False,False,True,False,False,False,False,True
1,2019,13451,145,58.9,1.0,0.0,0.0,0.0,0.0,0.0,...,False,False,True,False,False,False,False,False,False,True
2,2016,58264,30,61.7,1.7,0.0,0.0,0.0,0.0,0.0,...,False,False,True,False,False,True,False,False,False,False
3,2015,6147,145,47.9,2.0,0.0,0.0,1.0,0.0,0.0,...,False,False,False,False,True,False,False,False,False,True
4,2019,7000,145,51.4,1.6,0.0,0.0,0.0,0.0,0.0,...,False,False,True,False,False,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39080,2016,48868,30,61.7,1.7,0.0,0.0,0.0,0.0,0.0,...,False,False,True,False,False,True,False,False,False,False
39081,2019,8300,135,74.3,1.8,0.0,0.0,0.0,0.0,0.0,...,False,True,False,False,False,False,False,True,False,False
39082,2019,16916,145,65.7,2.0,0.0,0.0,0.0,0.0,0.0,...,False,True,False,False,False,True,False,False,False,False
39083,2014,27782,160,50.4,3.0,0.0,0.0,0.0,0.0,0.0,...,False,True,False,False,False,True,False,False,False,False


# 모델 구축

In [45]:
# 회귀모델 정의
model = DecisionTreeRegressor(random_state = 84)

In [46]:
# 모델 학습
model.fit(train_x, train_y)

DecisionTreeRegressor(random_state=84)

In [47]:
# 모델 성과 평가 --> 이건 학습에 사용한 데이터로 또 평가를 진행해서 성능이 너무 좋게 나옴
preds_train = model.predict(train_x)
print("RMSE :", np.sqrt(mean_squared_error(train_y, preds_train)))
print("R2 :", r2_score(train_y, preds_train))

RMSE : 162.17621870535754
R2 : 0.9997269779539487


In [48]:
# 추론
preds = model.predict(test_x)

In [49]:
# 제출
submission = pd.read_csv('./sample_submission.csv')
submission['price'] = preds

In [50]:
submission

,id,price
0,69940,19799.0
1,44315,15400.0
2,41540,12995.0
3,18443,12330.0
4,52142,18000.0
...,...,...
39080,13458,11950.0
39081,96993,23990.0
39082,43124,24000.0
39083,36905,16999.0


In [31]:
submission.to_csv('./submit.csv', index = False)